# ViewCastLK title scoring — Step 0.1: build the complete manifest

This component creates the repository-backed input file for Gemini title scoring. It contains every video exactly once and does **not** call Gemini yet.

The manifest columns are:

- `video_id`: stable key used to merge scores back into the master table;
- `title`: the exact UTF-8 title sent for scoring; and
- `title_sha256`: detects a changed title or an incorrect later merge.

The file is sorted by `video_id`, so rebuilding it from unchanged source data produces identical content.

In [ ]:
import hashlib
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if not (project_root / "Dataset").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from scripts.build_title_scoring_manifest import (
    MANIFEST_COLUMNS,
    build_manifest,
    read_source_titles,
    title_sha256,
)

source_path = project_root / "Dataset" / "viewcastlk_training_table.csv"
manifest_path = project_root / "Dataset" / "title_scoring_manifest.csv"

In [ ]:
manifest = build_manifest(source_path, manifest_path)

manifest_summary = pd.DataFrame([{
    "output_file": str(manifest_path),
    "rows": len(manifest),
    "unique_video_ids": manifest["video_id"].nunique(),
    "empty_titles": int(manifest["title"].str.strip().eq("").sum()),
    "duplicate_video_ids": int(manifest["video_id"].duplicated().sum()),
    "file_size_mb": manifest_path.stat().st_size / 1_000_000,
}])
display(manifest_summary)

## Actual manifest output

These are real records from the generated scoring manifest.

In [ ]:
display(manifest.head(10))
display(manifest.sample(10, random_state=42).sort_values("video_id"))

## Manifest tests

These tests verify one-to-one mapping, exact title preservation, valid hashes, and deterministic rebuilding. Any failure stops execution.

In [ ]:
source_titles = read_source_titles(source_path)
source_by_id = source_titles.set_index("video_id")["title"].sort_index()
manifest_by_id = manifest.set_index("video_id").sort_index()

file_hash_before_rebuild = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
rebuilt_manifest = build_manifest(source_path, manifest_path)
file_hash_after_rebuild = hashlib.sha256(manifest_path.read_bytes()).hexdigest()

recomputed_hashes = manifest["title"].map(title_sha256)
checks = {
    "manifest has expected columns": list(manifest.columns) == MANIFEST_COLUMNS,
    "manifest row count matches master": len(manifest) == len(source_titles),
    "every video_id is unique": manifest["video_id"].is_unique,
    "no video_id is empty": manifest["video_id"].str.strip().ne("").all(),
    "no title is empty": manifest["title"].str.strip().ne("").all(),
    "video_id set exactly matches master": set(manifest["video_id"]) == set(source_titles["video_id"]),
    "every title exactly matches master by video_id": manifest_by_id["title"].equals(source_by_id),
    "every title hash is correct": manifest["title_sha256"].equals(recomputed_hashes.rename("title_sha256")),
    "rebuild returns identical dataframe": rebuilt_manifest.equals(manifest),
    "rebuild produces identical file bytes": file_hash_before_rebuild == file_hash_after_rebuild,
}

manifest_test_results = pd.DataFrame([
    {"test": test_name, "result": "PASS" if passed else "FAIL"}
    for test_name, passed in checks.items()
])
display(manifest_test_results)

failed_manifest_tests = manifest_test_results.loc[manifest_test_results["result"] == "FAIL"]
assert failed_manifest_tests.empty, f"Manifest tests failed:\n{failed_manifest_tests.to_string(index=False)}"

## Manifest checkpoint — confirmed

The validated manifest above is the input to the bounded API pilot.

# Step 0.2: pinned 50-title Gemini pilot

This component scores exactly the first 50 deterministic manifest rows. It pins one stable model, prompt hash, and schema version. Completed IDs are skipped, so **Run All does not call Gemini again after the pilot is complete**.

No fallback model exists in the scoring code. A quota response leaves remaining rows pending for a later run.

In [ ]:
import os

from dotenv import load_dotenv

from scripts.score_video_titles import (
    MODEL_ID,
    PROMPT_SHA256,
    PROMPT_VERSION,
    SCHEMA_VERSION,
    read_scores,
    score_rows,
)

PILOT_SIZE = 50
REQUESTS_PER_MINUTE = 8.0
scores_path = project_root / "Dataset" / "title_scores.csv"
pilot_manifest = manifest.head(PILOT_SIZE).copy()

scoring_configuration = pd.DataFrame([{
    "model_id": MODEL_ID,
    "prompt_version": PROMPT_VERSION,
    "prompt_sha256": PROMPT_SHA256,
    "schema_version": SCHEMA_VERSION,
    "pilot_rows": len(pilot_manifest),
    "output_file": str(scores_path),
}])
display(scoring_configuration)

In [ ]:
load_dotenv(project_root / ".env")
api_key = os.getenv("GEMINI_API_KEY", "")
assert api_key.strip(), "GEMINI_API_KEY is not configured in .env"

pilot_run_summary = score_rows(
    pilot_manifest,
    scores_path,
    api_key,
    requests_per_minute=REQUESTS_PER_MINUTE,
)
display(pd.DataFrame([pilot_run_summary]))

# Remove the secret from notebook memory immediately after constructing the client calls.
del api_key

## Actual Gemini scores

The table contains every title in the 50-row pilot and its four real Gemini scores.

In [ ]:
scores = read_scores(scores_path)
pilot_scored = pilot_manifest.merge(
    scores,
    on=["video_id", "title_sha256"],
    how="left",
    validate="one_to_one",
)
score_columns = [
    "title_urgency",
    "title_emotional_appeal",
    "title_seriousness",
    "title_curiosity_gap",
]
for column in score_columns:
    pilot_scored[column] = pd.to_numeric(pilot_scored[column], errors="coerce")

display(pilot_scored[["video_id", "title", *score_columns]])
display(pilot_scored[score_columns].describe())

## Pilot integrity tests

These tests confirm complete coverage, correct title mapping, valid score ranges, and one consistent model/prompt/schema configuration.

In [ ]:
pilot_ids = set(pilot_manifest["video_id"].astype(str))
pilot_score_rows = scores.loc[scores["video_id"].astype(str).isin(pilot_ids)].copy()

checks = {
    "exactly 50 pilot rows are scored": len(pilot_score_rows) == PILOT_SIZE,
    "every pilot ID has a score": pilot_scored["model_id"].notna().all(),
    "score file has no duplicate video IDs": scores["video_id"].is_unique,
    "only the pinned model ID was used": set(pilot_score_rows["model_id"]) == {MODEL_ID},
    "only one returned API model version exists": pilot_score_rows["api_model_version"].nunique() == 1,
    "only the fixed prompt version was used": set(pilot_score_rows["prompt_version"]) == {PROMPT_VERSION},
    "only the fixed prompt hash was used": set(pilot_score_rows["prompt_sha256"]) == {PROMPT_SHA256},
    "only the fixed schema version was used": set(pilot_score_rows["schema_version"]) == {SCHEMA_VERSION},
    "title hashes match the manifest": pilot_scored["model_id"].notna().all(),
    "all four scores are integers from 0 to 10": all(
        pilot_scored[column].notna().all()
        and pilot_scored[column].between(0, 10).all()
        and pilot_scored[column].mod(1).eq(0).all()
        for column in score_columns
    ),
}

pilot_test_results = pd.DataFrame([
    {"test": test_name, "result": "PASS" if passed else "FAIL"}
    for test_name, passed in checks.items()
])
display(pilot_test_results)

failed_pilot_tests = pilot_test_results.loc[pilot_test_results["result"] == "FAIL"]
assert failed_pilot_tests.empty, f"Pilot tests failed:\n{failed_pilot_tests.to_string(index=False)}"

## Stop here — 50-title pilot checkpoint

Confirm the actual title scores, their distributions, and the test table before adding the scheduled GitHub Actions backfill.